Groundwater | Case Study

# Topic 5 : Model Calibration

Dr. Xiang-Zhao Kong & Dr. Beatrice Marti & Louise Noel du Payrat

In [ ]:
# Import libraries
import sys
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import geopandas as gpd

import flopy
from flopy.utils import HeadFile

from IPython.display import display

# Load local modules
sys.path.append('../SUPPORT_REPO/src')
sys.path.append('../SUPPORT_REPO/src/scripts/scripts_exercises')

from data_utils import (
    download_named_file,
    get_default_data_folder,
)
import print_images as du

# Set up plotting defaults
plt.rcParams['figure.figsize'] = (10, 8)
plt.rcParams['font.size'] = 12

## Introduction

In the previous notebooks, we defined our modeling objectives (Notebook 1), developed a perceptual model of the Limmat Valley aquifer (Notebook 2), chose the conceptual model (Notebook 3), and implemented the numerical model using MODFLOW-NWT (Notebook 4). 

Now we move to **Step 5: Model Calibration** - the process of adjusting model parameters so that simulated outputs match observed measurements. This is a critical step that bridges the gap between our theoretical model and the real-world system.

### What is Calibration?

Calibration is the process of adjusting model parameters within physically reasonable bounds to minimize the difference between simulated and observed values. In groundwater modeling, we typically calibrate against:

- **Hydraulic heads** (groundwater levels) at observation wells
- **Groundwater fluxes** (e.g., baseflow to rivers, spring discharge)
- **Contaminant concentrations** (for transport models)

### Calibration Workflow

1. **Define calibration targets**: Select observation data and define metrics
2. **Run the model**: Execute with initial parameter estimates
3. **Compare outputs**: Calculate residuals (simulated - observed)
4. **Adjust parameters**: Modify parameters to reduce residuals
5. **Iterate**: Repeat until acceptable fit is achieved
6. **Evaluate**: Check physical plausibility and water balance

> **Important: Calibration vs. Validation**
>
> We should **never** use all available data for calibration. A portion of the data must be reserved for independent validation (Step 6). This ensures the model has predictive capability, not just good fit to training data.
>
> Common splits are 75%/25% or 70%/30% for calibration/validation.

## 1 - Loading the Calibrated Model

We start by loading the MODFLOW-NWT model that we built in Notebook 4. The model files are stored in the workspace directory.

In [ ]:
# Define model paths
model_name = 'limmat_valley_model_nwt'
workspace = os.path.join(get_default_data_folder(), 'calibration')

# Check if model directory exists
if not os.path.exists(workspace):
    # Create the directory if it doesn't exist
    os.makedirs(workspace)
    
    # Try to download and extract baseline model
    print("\nAttempting to download baseline model...")
    baseline_path = download_named_file('baseline_model', data_type='calibration')

    # Extract if it's a zip file
    if baseline_path.endswith('.zip'):
        import zipfile
        extract_dir = os.path.dirname(baseline_path)
        with zipfile.ZipFile(baseline_path, 'r') as zip_ref:
            zip_ref.extractall(extract_dir)
        print(f"Extracted to: {extract_dir}")
        
        # Update workspace path
        #workspace = os.path.join(extract_dir, model_name)

else:
    print(f"Model workspace found at: {workspace}")

In [ ]:
# Load the MODFLOW-NWT model
mf = flopy.modflow.Modflow.load(
    f"{model_name}.nam",
    model_ws=workspace,
    exe_name='mfnwt',
    version='mfnwt',
    verbose=False
)

print(f"Model loaded: {mf.name}")
print(f"Number of layers: {mf.nlay}")
print(f"Number of rows: {mf.nrow}")
print(f"Number of columns: {mf.ncol}")
print(f"Packages: {[pkg.name[0] for pkg in mf.packagelist]}")

# IMPORTANT: Restore grid transformation parameters
# When loading from files, FloPy doesn't automatically restore the grid rotation.
# These values must match what was used when the model was created in Notebook 4.
# The grid was rotated 30 degrees and positioned at a specific origin in Swiss LV95 coordinates.

# Grid rotation angle (from Notebook 4: grid_rotation_angle = 30)
grid_rotation_angle = 30  # degrees

# Read the grid origin from the DIS package if available, or set manually
# The DIS package stores xul, yul (upper-left corner) and rotation
if hasattr(mf, 'dis'):
    # Try to get from DIS package attributes
    xul = getattr(mf.dis, 'xul', None)
    yul = getattr(mf.dis, 'yul', None)
    rotation = getattr(mf.dis, 'rotation', None)
    
    if xul is not None and yul is not None:
        print(f"\nGrid parameters from DIS package:")
        print(f"  xul: {xul}, yul: {yul}, rotation: {rotation}")
    else:
        # Set manually based on Notebook 4 values
        # These are approximate - adjust based on your actual model
        print("\nGrid parameters not found in DIS - setting manually...")

# Set the modelgrid transformation parameters
# Note: FloPy's modelgrid uses xoffset/yoffset (lower-left) and angrot
# We need to convert from xul/yul (upper-left) to xoffset/yoffset (lower-left)

# Calculate grid dimensions
total_width = np.sum(mf.dis.delr.array)  # Total width in x (columns)
total_height = np.sum(mf.dis.delc.array)  # Total height in y (rows)

# For now, let's check what values are currently set and print diagnostic info
print(f"\nCurrent modelgrid parameters:")
print(f"  xoffset: {mf.modelgrid.xoffset}")
print(f"  yoffset: {mf.modelgrid.yoffset}")
print(f"  angrot: {mf.modelgrid.angrot}")
print(f"  Grid dimensions: {total_width:.1f} x {total_height:.1f} m")

# If the grid is not rotated, we need to set the transformation
# This requires knowing the correct origin coordinates from Notebook 4
if mf.modelgrid.angrot == 0:
    print("\n⚠ Grid rotation is 0 - transformation parameters need to be set!")
    print("  Please run the cell below to set the correct grid transformation.")

In [ ]:
# Set grid transformation parameters for the rotated grid
# These values MUST match what was used in Notebook 4 when creating the model
#
# From Notebook 4, the StructuredGrid was created with:
#   xoff=xmin_original  (lower-left x)
#   yoff=ymin_original  (lower-left y)
#   angrot=-grid_rotation_angle  (negative rotation angle)

grid_rotation_angle = 30  # degrees as defined in Notebook 4

print("Calculating grid transformation from boundary file (same as Notebook 4)...")

from shapely.affinity import rotate
from shapely.geometry import Point

boundary_file = download_named_file('model_boundary', data_type='gis')
boundary_gdf = gpd.read_file(boundary_file)
boundary_geom = boundary_gdf.geometry.iloc[0]

# Rotate boundary to find grid extent (same as in Notebook 4)
origin_rotation = Point(0, 0)
boundary_rotated = rotate(boundary_geom, grid_rotation_angle, origin=origin_rotation)

# Get bounds in rotated coordinate system
minx_rot, miny_rot, maxx_rot, maxy_rot = boundary_rotated.bounds

# Add buffer (same as in Notebook 4)
buffer_m = 50
minx_rot -= buffer_m
miny_rot -= buffer_m
maxx_rot += buffer_m
maxy_rot += buffer_m

# Create corner points in rotated system and transform back to real-world coords
min_point_rotated = Point(minx_rot, miny_rot)
max_point_rotated = Point(maxx_rot, maxy_rot)

min_point_original = rotate(min_point_rotated, -grid_rotation_angle, origin=origin_rotation)
max_point_original = rotate(max_point_rotated, -grid_rotation_angle, origin=origin_rotation)

# Grid origin in real-world coordinates - LOWER-LEFT corner
xmin_original = min_point_original.x
ymin_original = min_point_original.y
xmax_original = max_point_original.x
ymax_original = max_point_original.y

print(f"Grid corners in real-world coordinates:")
print(f"  xmin: {xmin_original:.2f}, ymin: {ymin_original:.2f}")
print(f"  xmax: {xmax_original:.2f}, ymax: {ymax_original:.2f}")

# Set the modelgrid transformation - EXACTLY as in Notebook 4
mf.modelgrid.set_coord_info(
    xoff=xmin_original,           # Lower-left x
    yoff=ymin_original,           # Lower-left y
    angrot=-grid_rotation_angle   # Negative rotation (same as Notebook 4)
)

print(f"\nGrid transformation set:")
print(f"  xoffset: {mf.modelgrid.xoffset:.2f}")
print(f"  yoffset: {mf.modelgrid.yoffset:.2f}")
print(f"  angrot: {mf.modelgrid.angrot} degrees")
print(f"  Grid extent: {mf.modelgrid.extent}")

## 2 - Loading Observation Data

We use groundwater level observations from the AWEL (Amt fur Abfall, Wasser, Energie und Luft) monitoring network, which we introduced in Notebook 2 (Chapter 6: Monitoring the Limmat Valley Aquifer).

### 2.1 - Groundwater Level Time Series

The time series data contains daily groundwater levels from multiple observation wells.

In [ ]:
# Download groundwater time series data
gw_ts_path = download_named_file(
    name='groundwater_timeseries',
    data_type='time_series'
)

# Load the data
gw_ts_raw = pd.read_csv(gw_ts_path)

# Process the data - keep coordinates from the raw data
gw_ts = gw_ts_raw[['date', 'value', 'well_id', 'x_coord', 'y_coord']].copy()
gw_ts['date'] = pd.to_datetime(gw_ts['date'], errors='coerce')
gw_ts = gw_ts.drop_duplicates()
gw_ts = gw_ts.sort_values('date')
# Make sure well_id is string
gw_ts['well_id'] = gw_ts['well_id'].astype(str)

# Drop rows with invalid well_ids (e.g., NaN and 'nan')
gw_ts = gw_ts[~gw_ts['well_id'].isin(['nan', 'NaN', ''])]
# Drop rows with invalid dates
gw_ts = gw_ts.dropna(subset=['date'])

# Standardize well IDs (B53 and B5-3 are the same well)
gw_ts.loc[gw_ts['well_id'] == 'B53', 'well_id'] = 'B5-3'

print(f"Loaded {len(gw_ts)} observations from {gw_ts['well_id'].nunique()} wells")
print(f"Date range: {gw_ts['date'].min()} to {gw_ts['date'].max()}")
print(f"\nWells: {sorted(gw_ts['well_id'].unique())}")

In [ ]:
# Calculate summary statistics for each well
well_stats = gw_ts.groupby('well_id')['value'].agg(
    ['mean', 'std', 'min', 'max', 'count']
).round(2)
well_stats['range'] = well_stats['max'] - well_stats['min']
well_stats = well_stats.sort_values('mean', ascending=False)

print("Summary statistics for observation wells (head in m a.s.l.):")
display(well_stats)

### 2.2 - Observation Well Locations

The groundwater time series data already includes the spatial coordinates (x_coord, y_coord) for each observation well. We can use these directly to find the corresponding model grid cells.

In [ ]:
# Extract unique well locations from the time series data
well_locations = gw_ts.groupby('well_id').agg({
    'x_coord': 'first',
    'y_coord': 'first'
}).reset_index()

# Convert coordinates to numeric (they may be strings)
well_locations['x_coord'] = pd.to_numeric(well_locations['x_coord'], errors='coerce')
well_locations['y_coord'] = pd.to_numeric(well_locations['y_coord'], errors='coerce')

print(f"Found {len(well_locations)} unique observation wells with coordinates:")
display(well_locations)

### 2.3 - Matching Wells to Model Grid

We need to find which model cell each observation well falls into. This allows us to extract simulated heads at the correct locations.

In [ ]:
# First, let's check the model grid properties
print("Model Grid Properties:")
print(f"  X offset (xoffset): {mf.modelgrid.xoffset}")
print(f"  Y offset (yoffset): {mf.modelgrid.yoffset}")
print(f"  Rotation angle (angrot): {mf.modelgrid.angrot} degrees")
print(f"  Grid extent: {mf.modelgrid.extent}")
print(f"  Grid shape: {mf.modelgrid.nrow} rows x {mf.modelgrid.ncol} cols")

# Check well coordinates vs grid extent
print(f"\nWell coordinate ranges:")
print(f"  X: {well_locations['x_coord'].min():.0f} to {well_locations['x_coord'].max():.0f}")
print(f"  Y: {well_locations['y_coord'].min():.0f} to {well_locations['y_coord'].max():.0f}")

In [ ]:
def find_cell_for_point(modelgrid, x, y):
    """
    Find the model cell (row, col) containing a point.
    Handles rotated grids by converting to local coordinates first.
    
    Parameters
    ----------
    modelgrid : flopy.discretization.StructuredGrid
        The model grid object
    x, y : float
        Coordinates of the point in real-world (Swiss LV95) coordinates
        
    Returns
    -------
    tuple or None
        (row, col) if point is within grid, None otherwise
    """
    try:
        # For rotated grids, we need to convert to local coordinates first
        # Local coordinates are in the model's own coordinate system (unrotated)
        if modelgrid.angrot != 0:
            local_x, local_y = modelgrid.get_local_coords(x, y)
        else:
            local_x, local_y = x - modelgrid.xoffset, y - modelgrid.yoffset
        
        # In local coordinates, find which row/col the point falls into
        # Row increases downward (decreasing y), col increases rightward (increasing x)
        delr = modelgrid.delr  # column widths (along rows, i.e., x-direction)
        delc = modelgrid.delc  # row heights (along columns, i.e., y-direction)
        
        # Calculate cumulative distances
        x_edges = np.concatenate([[0], np.cumsum(delr)])  # column edges
        y_edges = np.concatenate([[0], np.cumsum(delc)])  # row edges (from top)
        
        # Grid total dimensions in local coords
        total_x = x_edges[-1]
        total_y = y_edges[-1]
        
        # Check if point is within grid bounds
        if local_x < 0 or local_x > total_x:
            return None
        if local_y < 0 or local_y > total_y:
            return None
        
        # Find column index (x increases to the right)
        col = np.searchsorted(x_edges, local_x, side='right') - 1
        col = min(col, modelgrid.ncol - 1)  # Clamp to valid range
        
        # Find row index (y increases upward in local coords, but rows increase downward)
        # local_y is distance from origin (bottom-left), rows count from top
        row = np.searchsorted(y_edges, total_y - local_y, side='right') - 1
        row = min(row, modelgrid.nrow - 1)  # Clamp to valid range
        
        return (int(row), int(col))
    except Exception as e:
        print(f"Error finding cell for point ({x}, {y}): {e}")
        return None

# Test the function with the first well location
test_x = well_locations['x_coord'].iloc[0]
test_y = well_locations['y_coord'].iloc[0]
cell = find_cell_for_point(mf.modelgrid, test_x, test_y)
print(f"Test point ({test_x:.0f}, {test_y:.0f}) -> Cell: {cell}")

# Also show the local coordinates for debugging
if mf.modelgrid.angrot != 0:
    local_x, local_y = mf.modelgrid.get_local_coords(test_x, test_y)
    print(f"  Local coordinates: ({local_x:.1f}, {local_y:.1f})")
    print(f"  Grid dimensions: {np.sum(mf.modelgrid.delr):.1f} x {np.sum(mf.modelgrid.delc):.1f}")

In [ ]:
# Find model cells for all observation wells
obs_wells = []
wells_outside = []
wells_inactive = []

print("Processing observation wells:")
print("-" * 80)

for idx, row in well_locations.iterrows():
    well_id = row['well_id']
    x, y = row['x_coord'], row['y_coord']
    
    # Skip wells with missing coordinates
    if pd.isna(x) or pd.isna(y):
        print(f"  {well_id}: Skipped - missing coordinates")
        continue
    
    # Get local coordinates for debugging
    if mf.modelgrid.angrot != 0:
        local_x, local_y = mf.modelgrid.get_local_coords(x, y)
    else:
        local_x, local_y = x - mf.modelgrid.xoffset, y - mf.modelgrid.yoffset
    
    cell = find_cell_for_point(mf.modelgrid, x, y)
    if cell is not None:
        i, j = cell
        # Check if cell is active (ibound > 0)
        if hasattr(mf, 'bas6') and mf.bas6.ibound.array[0, i, j] > 0:
            obs_wells.append({
                'well_id': well_id,
                'x': x,
                'y': y,
                'local_x': local_x,
                'local_y': local_y,
                'row': i,
                'col': j,
                'layer': 0
            })
            print(f"  {well_id}: ({x:.0f}, {y:.0f}) -> local ({local_x:.1f}, {local_y:.1f}) -> cell ({i}, {j}) - ACTIVE")
        else:
            wells_inactive.append(well_id)
            print(f"  {well_id}: ({x:.0f}, {y:.0f}) -> local ({local_x:.1f}, {local_y:.1f}) -> cell ({i}, {j}) - INACTIVE")
    else:
        wells_outside.append(well_id)
        print(f"  {well_id}: ({x:.0f}, {y:.0f}) -> local ({local_x:.1f}, {local_y:.1f}) -> OUTSIDE GRID")

obs_wells_df = pd.DataFrame(obs_wells)

print("-" * 80)
print(f"\nSummary:")
print(f"  Wells within active model domain: {len(obs_wells_df)}")
print(f"  Wells outside model domain: {len(wells_outside)} - {wells_outside}")
print(f"  Wells in inactive cells: {len(wells_inactive)} - {wells_inactive}")

if len(obs_wells_df) > 0:
    print("\nObservation wells matched to grid cells:")
    display(obs_wells_df[['well_id', 'x', 'y', 'row', 'col']])
else:
    print("\nWARNING: No observation wells found within the active model domain!")
    print("Check the visualization in the next cell to diagnose the issue.")
    print("\nDebugging info:")
    print(f"  Grid rotation: {mf.modelgrid.angrot} degrees")
    print(f"  Grid offset: ({mf.modelgrid.xoffset}, {mf.modelgrid.yoffset})")
    print(f"  Grid dimensions (local): {np.sum(mf.modelgrid.delr):.1f} x {np.sum(mf.modelgrid.delc):.1f}")

In [ ]:
# Visualize well locations relative to model grid
fig, ax = plt.subplots(figsize=(12, 10))

# Check if grid has rotation - if not, we may need to set it manually
print(f"Grid rotation: {mf.modelgrid.angrot} degrees")
print(f"Grid offset: ({mf.modelgrid.xoffset}, {mf.modelgrid.yoffset})")

# Plot the model grid with ibound
# PlotMapView will show the grid in real-world (rotated) coordinates
pmv = flopy.plot.PlotMapView(model=mf, ax=ax)
ibound_plot = pmv.plot_ibound()
pmv.plot_grid(linewidth=0.3, alpha=0.5)

# The well coordinates are in Swiss LV95 (real-world coordinates)
# They should plot correctly on the rotated grid
ax.scatter(well_locations['x_coord'], well_locations['y_coord'], 
           c='red', s=100, marker='o', label='Observation wells', zorder=5, edgecolors='black')

# Add well labels
for idx, row in well_locations.iterrows():
    ax.annotate(row['well_id'], (row['x_coord'], row['y_coord']), 
                xytext=(5, 5), textcoords='offset points', fontsize=8)

# If wells don't appear on the grid, print diagnostic info
grid_extent = mf.modelgrid.extent
print(f"\nGrid extent (in real-world coords): {grid_extent}")
print(f"Well X range: {well_locations['x_coord'].min():.0f} to {well_locations['x_coord'].max():.0f}")
print(f"Well Y range: {well_locations['y_coord'].min():.0f} to {well_locations['y_coord'].max():.0f}")

ax.set_xlabel('X [m] - Swiss LV95')
ax.set_ylabel('Y [m] - Swiss LV95')
ax.set_title('Observation Well Locations vs Model Grid (Real-World Coordinates)')
ax.legend()
ax.set_aspect('equal')
plt.tight_layout()
plt.show()

# If the grid appears unrotated but wells are in rotated coords (or vice versa),
# print a warning
if mf.modelgrid.angrot == 0:
    print("\n⚠ WARNING: Grid rotation is 0 degrees.")
    print("  If the model was built with rotation, the grid parameters may not have")
    print("  been preserved when loading. You may need to manually set the grid")
    print("  transformation parameters (xoffset, yoffset, angrot) after loading.")

In [ ]:
# Calculate mean heads for calibration
# We use a recent period (e.g., 2015-2020) to match current conditions
calibration_period = (gw_ts['date'] >= '2015-01-01') & (gw_ts['date'] <= '2020-12-31')
gw_ts_calib = gw_ts[calibration_period].copy()

# Calculate mean head per well
mean_heads = gw_ts_calib.groupby('well_id')['value'].mean().round(2)
mean_heads.name = 'observed_head'

print(f"Mean observed heads (2015-2020):")
print(mean_heads)

## 3 - Calibration Metrics

We define several metrics to quantify the fit between simulated and observed heads.

In [ ]:
def calculate_calibration_metrics(observed, simulated):
    """
    Calculate calibration performance metrics.
    
    Parameters
    ----------
    observed : array-like
        Observed values
    simulated : array-like
        Simulated values
        
    Returns
    -------
    dict
        Dictionary of metrics
    """
    obs = np.array(observed)
    sim = np.array(simulated)
    
    # Remove any NaN values
    mask = ~(np.isnan(obs) | np.isnan(sim))
    obs = obs[mask]
    sim = sim[mask]
    
    n = len(obs)
    if n == 0:
        return None
    
    residuals = sim - obs
    
    # Mean Error (Bias)
    me = np.mean(residuals)
    
    # Mean Absolute Error
    mae = np.mean(np.abs(residuals))
    
    # Root Mean Square Error
    rmse = np.sqrt(np.mean(residuals**2))
    
    # Coefficient of determination (R^2)
    ss_res = np.sum(residuals**2)
    ss_tot = np.sum((obs - np.mean(obs))**2)
    r2 = 1 - (ss_res / ss_tot) if ss_tot > 0 else 0
    
    # Normalized RMSE (as percentage of observed range)
    obs_range = obs.max() - obs.min()
    nrmse = (rmse / obs_range * 100) if obs_range > 0 else 0
    
    return {
        'n': n,
        'ME (bias)': round(me, 3),
        'MAE': round(mae, 3),
        'RMSE': round(rmse, 3),
        'NRMSE (%)': round(nrmse, 1),
        'R2': round(r2, 3)
    }

# Test with dummy data
test_obs = [100, 101, 102, 103]
test_sim = [100.5, 101.2, 101.8, 103.1]
print("Test metrics:", calculate_calibration_metrics(test_obs, test_sim))

### Interpretation of Metrics

| Metric | Description | Good Value |
|--------|-------------|------------|
| **ME (Bias)** | Mean error - positive means model over-predicts | Close to 0 |
| **MAE** | Mean absolute error | < 1-2 m for regional models |
| **RMSE** | Root mean square error - sensitive to outliers | < 1-2 m for regional models |
| **NRMSE** | Normalized RMSE as % of observed range | < 10% is good |
| **R^2** | Coefficient of determination | > 0.9 is good |

## 4 - Initial Model Run

Let's run the model with the initial (uncalibrated) parameters and compare against observations.

In [ ]:
# Write input files and run the model
mf.write_input()
success, buff = mf.run_model(silent=True)

if success:
    print("Model run completed successfully!")
else:
    print("Model run failed. Check the listing file for errors.")
    # Print last few lines of buffer for debugging
    print("\nLast lines of output:")
    for line in buff[-10:]:
        print(line)

In [ ]:
# Load simulated heads
head_file = os.path.join(workspace, f"{model_name}.hds")

if os.path.exists(head_file):
    hds = HeadFile(head_file)
    head = hds.get_data(kstpkper=(0, 0))  # Steady state: first time step, first stress period
    print(f"Head array shape: {head.shape}")
    print(f"Head range: {np.nanmin(head):.2f} to {np.nanmax(head):.2f} m")
else:
    print(f"Head file not found: {head_file}")

In [ ]:
# Extract simulated heads at observation well locations
def extract_simulated_heads(head_array, obs_wells_df, layer=0):
    """
    Extract simulated heads at observation well locations.
    
    Parameters
    ----------
    head_array : np.ndarray
        3D array of simulated heads [nlay, nrow, ncol]
    obs_wells_df : pd.DataFrame
        DataFrame with observation well locations (row, col columns)
    layer : int
        Layer index to extract from
        
    Returns
    -------
    np.ndarray
        Array of simulated heads at observation locations
    """
    sim_heads = []
    for _, well in obs_wells_df.iterrows():
        i, j = int(well['row']), int(well['col'])
        h = head_array[layer, i, j]
        # Check for dry cells (MODFLOW uses large negative values)
        if h < -999:
            h = np.nan
        sim_heads.append(h)
    return np.array(sim_heads)

# Extract simulated heads (only if we have observation wells)
if 'head' in dir() and 'obs_wells_df' in dir() and len(obs_wells_df) > 0:
    obs_wells_df['simulated_head'] = extract_simulated_heads(head, obs_wells_df)
    print("Simulated heads extracted at observation wells:")
    display(obs_wells_df)
else:
    if 'obs_wells_df' not in dir() or len(obs_wells_df) == 0:
        print("No observation wells found in active model domain - cannot extract simulated heads.")
        print("Please check the well locations and model grid alignment.")
    elif 'head' not in dir():
        print("Head array not available - run the model first.")

## 5 - Calibration Assessment

Now let's compare simulated vs. observed heads and calculate performance metrics.

In [ ]:
# Create comparison DataFrame by matching well IDs between obs_wells_df and mean_heads
if 'obs_wells_df' in dir() and len(obs_wells_df) > 0:
    comparison_df = obs_wells_df.copy()
    
    # Match observed heads from mean_heads using well_id
    comparison_df['observed_head'] = comparison_df['well_id'].map(mean_heads)
    
    # Calculate residuals (simulated - observed)
    comparison_df['residual'] = comparison_df['simulated_head'] - comparison_df['observed_head']
    
    print("Comparison of simulated vs observed heads:")
    display(comparison_df)
    
    # Show matching statistics
    n_matched = comparison_df['observed_head'].notna().sum()
    n_total = len(comparison_df)
    print(f"\nMatched {n_matched} of {n_total} observation wells with mean head data")
    
    if n_matched > 0:
        # Calculate and display calibration metrics
        valid_mask = comparison_df['observed_head'].notna() & comparison_df['simulated_head'].notna()
        if valid_mask.any():
            metrics = calculate_calibration_metrics(
                comparison_df.loc[valid_mask, 'observed_head'].values,
                comparison_df.loc[valid_mask, 'simulated_head'].values
            )
            print(f"\nCalibration Metrics:")
            for key, value in metrics.items():
                print(f"  {key}: {value}")
else:
    print("No observation wells available for comparison.")

In [ ]:
def plot_calibration_scatter(observed, simulated, title="Calibration Results"):
    """
    Create a scatter plot of simulated vs observed heads.
    """
    fig, ax = plt.subplots(figsize=(8, 8))
    
    # Remove NaN values
    mask = ~(np.isnan(observed) | np.isnan(simulated))
    obs = np.array(observed)[mask]
    sim = np.array(simulated)[mask]
    
    if len(obs) == 0:
        print("No valid data points for plotting")
        return fig, ax
    
    # Plot points
    ax.scatter(obs, sim, s=100, alpha=0.7, edgecolors='black', linewidth=1)
    
    # Add 1:1 line
    min_val = min(obs.min(), sim.min())
    max_val = max(obs.max(), sim.max())
    margin = (max_val - min_val) * 0.05
    line_range = [min_val - margin, max_val + margin]
    ax.plot(line_range, line_range, 'k--', label='1:1 line', linewidth=2)
    
    # Add +/- 1m error bands
    ax.fill_between(line_range, 
                    [x - 1 for x in line_range], 
                    [x + 1 for x in line_range],
                    alpha=0.2, color='gray', label='+/- 1m')
    
    # Calculate and display metrics
    metrics = calculate_calibration_metrics(obs, sim)
    if metrics:
        text = f"n = {metrics['n']}\n"
        text += f"RMSE = {metrics['RMSE']:.2f} m\n"
        text += f"MAE = {metrics['MAE']:.2f} m\n"
        text += f"R² = {metrics['R2']:.3f}\n"
        text += f"Bias = {metrics['ME (bias)']:.2f} m"
        ax.text(0.05, 0.95, text, transform=ax.transAxes, fontsize=11,
                verticalalignment='top', bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))
    
    ax.set_xlabel('Observed Head (m a.s.l.)', fontsize=12)
    ax.set_ylabel('Simulated Head (m a.s.l.)', fontsize=12)
    ax.set_title(title, fontsize=14)
    ax.legend(loc='lower right')
    ax.set_aspect('equal')
    ax.grid(True, alpha=0.3)
    
    plt.tight_layout()
    return fig, ax

# Plot calibration results (if data available)
if 'comparison_df' in dir() and not comparison_df['observed_head'].isna().all():
    fig, ax = plot_calibration_scatter(
        comparison_df['observed_head'].values,
        comparison_df['simulated_head'].values,
        title="Figure 1: Initial Calibration - Simulated vs Observed Heads"
    )
    plt.show()
else:
    print("Observed head data not yet matched. See TODO above.")

In [ ]:
def plot_residual_map(modelgrid, obs_wells_df, head_array, title="Residual Map"):
    """
    Create a map showing spatial distribution of residuals.
    """
    fig, ax = plt.subplots(figsize=(12, 10))
    
    # Mask heads outside model area (where heads are < 0)
    head_masked = np.ma.masked_where(head_array[0] < 0, head_array[0])
    
    # Plot head distribution
    pmv = flopy.plot.PlotMapView(modelgrid=modelgrid, ax=ax)
    head_plot = pmv.plot_array(head_masked, cmap='viridis', alpha=0.7)
    plt.colorbar(head_plot, ax=ax, label='Simulated Head (m a.s.l.)', shrink=0.5)
    
    # Plot observation wells colored by residual
    if 'residual' in obs_wells_df.columns:
        valid_mask = ~obs_wells_df['residual'].isna()
        if valid_mask.any():
            scatter = ax.scatter(
                obs_wells_df.loc[valid_mask, 'x'],
                obs_wells_df.loc[valid_mask, 'y'],
                c=obs_wells_df.loc[valid_mask, 'residual'],
                cmap='RdBu_r',
                s=200,
                edgecolors='black',
                linewidth=2,
                vmin=-3, vmax=3,
                zorder=5
            )
            plt.colorbar(scatter, ax=ax, label='Residual (m)', shrink=0.5)
    
    ax.set_xlabel('X (m)')
    ax.set_ylabel('Y (m)')
    ax.set_title(title)
    ax.set_aspect('equal')
    
    plt.tight_layout()
    return fig, ax

# Plot residual map - use comparison_df which has the residual column
if 'head' in dir() and 'comparison_df' in dir():
    fig, ax = plot_residual_map(
        mf.modelgrid, 
        comparison_df,  # comparison_df has the residual column, obs_wells_df does not
        head,
        title="Figure 2: Spatial Distribution of Calibration Residuals"
    )
    plt.show()

## 6 - Manual Calibration

Based on the residual analysis, we can identify which parameters need adjustment. Common calibration parameters include:

1. **Hydraulic conductivity (K)** - affects head gradients and flow velocities
2. **Recharge rates** - affects overall water balance
3. **River bed conductance** - affects river-aquifer exchange
4. **Boundary condition heads** - affects heads near boundaries

### 6.1 - Parameter Sensitivity

Before adjusting parameters, let's understand which parameters have the most influence.

In [ ]:
# Get current parameter values
print("Current model parameters:")
print("=" * 50)

# Hydraulic conductivity (UPW or LPF package)
if hasattr(mf, 'upw'):
    hk = mf.upw.hk.array
    print(f"\nHydraulic Conductivity (K):")
    print(f"  Min: {np.nanmin(hk):.2e} m/day")
    print(f"  Max: {np.nanmax(hk):.2e} m/day")
    print(f"  Mean: {np.nanmean(hk):.2e} m/day")
elif hasattr(mf, 'lpf'):
    hk = mf.lpf.hk.array
    print(f"\nHydraulic Conductivity (K):")
    print(f"  Min: {np.nanmin(hk):.2e} m/day")
    print(f"  Max: {np.nanmax(hk):.2e} m/day")
    print(f"  Mean: {np.nanmean(hk):.2e} m/day")

# Recharge
if hasattr(mf, 'rch'):
    rch = mf.rch.rech.array
    print(f"\nRecharge:")
    print(f"  Mean: {np.nanmean(rch)*1000:.2f} mm/day")
    print(f"  Annual: {np.nanmean(rch)*365*1000:.0f} mm/year")

# River package
if hasattr(mf, 'riv'):
    print(f"\nRiver cells: {len(mf.riv.stress_period_data[0])}")

### 6.2 - Parameter Adjustment Strategy

Based on the residual patterns:

- **Systematic bias (all residuals positive/negative)**: Adjust recharge or boundary heads
- **Spatial gradient issues**: Adjust hydraulic conductivity or its spatial distribution
- **Local errors near rivers**: Adjust river bed conductance

In [ ]:
# Example: Adjust hydraulic conductivity by a factor
def run_calibration_trial(mf, k_factor=1.0, rch_factor=1.0):
    """
    Run a calibration trial with adjusted parameters.
    
    Parameters
    ----------
    mf : flopy.modflow.Modflow
        The model object
    k_factor : float
        Multiplier for hydraulic conductivity
    rch_factor : float
        Multiplier for recharge
        
    Returns
    -------
    np.ndarray or None
        Head array if successful, None if failed
    """
    import copy
    
    # Create a copy to avoid modifying original
    # Note: This is simplified - full implementation would create new model
    
    # Adjust K
    if hasattr(mf, 'upw'):
        original_hk = mf.upw.hk.array.copy()
        mf.upw.hk = original_hk * k_factor
    elif hasattr(mf, 'lpf'):
        original_hk = mf.lpf.hk.array.copy()
        mf.lpf.hk = original_hk * k_factor
    
    # Adjust recharge
    if hasattr(mf, 'rch'):
        original_rch = mf.rch.rech.array.copy()
        mf.rch.rech = original_rch * rch_factor
    
    # Run model
    mf.write_input()
    success, _ = mf.run_model(silent=True)
    
    # Restore original values
    if hasattr(mf, 'upw'):
        mf.upw.hk = original_hk
    elif hasattr(mf, 'lpf'):
        mf.lpf.hk = original_hk
    if hasattr(mf, 'rch'):
        mf.rch.rech = original_rch
    
    if success:
        hds = HeadFile(os.path.join(mf.model_ws, f"{mf.name}.hds"))
        return hds.get_data(kstpkper=(0, 0))
    return None

print("Calibration trial function defined.")
print("Use run_calibration_trial(mf, k_factor=1.5) to test with 50% higher K.")

## 7 - Calibration Results Summary

After completing the calibration process, summarize the results.

In [ ]:
# Summary table of calibration iterations
# This would be filled in during actual calibration
calibration_log = pd.DataFrame({
    'Iteration': [1],
    'K_factor': [1.0],
    'Rch_factor': [1.0],
    'RMSE (m)': [np.nan],
    'MAE (m)': [np.nan],
    'Bias (m)': [np.nan],
    'R2': [np.nan],
    'Notes': ['Initial run']
})

print("Calibration Log:")
display(calibration_log)

## 8 - Water Balance Check

A calibrated model should have a reasonable water balance. Let's check the model's water budget.

In [ ]:
# Read water balance from the listing file (more complete than .cbc file)
list_file = os.path.join(workspace, f"{model_name}.list")

if os.path.exists(list_file):
    # Use FloPy's built-in listing file budget parser
    mfl = flopy.utils.MfListBudget(list_file)
    
    # Get the budget data as a DataFrame
    budget_df = mfl.get_dataframes()[0]  # First dataframe has cumulative budget
    
    print("="*60)
    print("WATER BALANCE SUMMARY (from listing file)")
    print("="*60)
    
    print(f"\n{'COMPONENT':<25} {'IN (m³/day)':>15} {'OUT (m³/day)':>15}")
    print("-"*60)
    
    total_in = 0.0
    total_out = 0.0
    
    # Get the last row (final time step)
    last_row = budget_df.iloc[-1]
    
    # Budget terms come in pairs: TERM_IN and TERM_OUT
    # Get unique term names (without _IN or _OUT suffix)
    in_cols = [c for c in budget_df.columns if c.endswith('_IN')]
    terms = [c.replace('_IN', '') for c in in_cols]
    
    for term in terms:
        in_col = f"{term}_IN"
        out_col = f"{term}_OUT"
        
        inflow = last_row[in_col] if in_col in last_row else 0.0
        outflow = last_row[out_col] if out_col in last_row else 0.0
        
        total_in += inflow
        total_out += outflow
        
        # Only print if there's flow
        if inflow > 0.01 or outflow > 0.01:
            print(f"{term:<25} {inflow:>15.2f} {outflow:>15.2f}")
    
    print("-"*60)
    print(f"{'TOTAL':<25} {total_in:>15.2f} {total_out:>15.2f}")
    print("="*60)
    
    # Calculate percent discrepancy
    avg_flow = (total_in + total_out) / 2
    if avg_flow > 0:
        discrepancy = (total_in - total_out) / avg_flow * 100
        print(f"\nIN - OUT:            {total_in - total_out:>10.4f} m³/day")
        print(f"Percent Discrepancy: {discrepancy:>10.4f} %")
        if abs(discrepancy) < 1.0:
            print("Water balance is acceptable (< 1% error)")
        else:
            print("WARNING: Water balance error exceeds 1%")
else:
    print(f"Listing file not found: {list_file}")

## 9 - Discussion

### Key Takeaways

1. **Calibration is iterative**: Multiple rounds of parameter adjustment are typically needed
2. **Physical plausibility matters**: Parameters must stay within realistic bounds
3. **Non-uniqueness**: Different parameter combinations can produce similar fits
4. **Trade-offs**: Improving fit at one location may worsen it elsewhere

### Limitations of Manual Calibration

- Time-consuming for many parameters
- Difficult to explore full parameter space
- Subjective decisions about which parameters to adjust

### When to Use Automated Calibration

For complex models with many parameters, consider tools like:
- **PEST** (Parameter ESTimation): Industry standard for groundwater model calibration
- **pyEMU**: Python interface to PEST, useful for uncertainty analysis

These tools systematically search the parameter space and provide uncertainty estimates.

## 10 - Automatic Calibration with PEST++

While manual calibration provides valuable insight into model behavior, it becomes impractical for models with many parameters. **Automatic calibration** tools use mathematical optimization algorithms to systematically adjust parameters and minimize the difference between simulated and observed values.

### 10.1 - Why Automatic Calibration?

| Aspect | Manual Calibration | Automatic Calibration |
|--------|-------------------|----------------------|
| **Parameters** | Feasible for 5-10 | Handles 100s-1000s |
| **Reproducibility** | Subjective | Objective, documented |
| **Uncertainty** | Qualitative | Quantitative estimates |
| **Time** | Days to weeks | Hours to days |
| **Insight** | High (if done well) | Requires interpretation |

### 10.2 - PEST++ Overview

**PEST++** is the modern, open-source successor to PEST (Parameter ESTimation), the industry-standard tool for model calibration. It is developed and maintained by the USGS.

**Key Features:**
- **Gradient-based optimization**: Efficiently finds parameter values that minimize objective function
- **Regularization**: Prevents overfitting by penalizing unrealistic parameter distributions
- **Parallel computing**: Runs model evaluations simultaneously on multiple cores
- **Uncertainty quantification**: Provides parameter and prediction uncertainty estimates
- **Global sensitivity analysis**: Identifies which parameters most influence model outputs

**PEST++ Program Suite:**
| Program | Purpose |
|---------|---------|
| `pestpp-glm` | Gradient-based parameter estimation (most common) |
| `pestpp-ies` | Iterative Ensemble Smoother for uncertainty analysis |
| `pestpp-sen` | Global sensitivity analysis (Morris method) |
| `pestpp-opt` | Optimization under uncertainty |
| `pestpp-sqp` | Sequential quadratic programming |

### 10.3 - Key Concepts in Automatic Calibration

#### The Objective Function

PEST++ minimizes an **objective function** (Φ) that quantifies model-to-measurement misfit:

$$\Phi = \sum_{i=1}^{n} w_i (y_i^{obs} - y_i^{sim})^2$$

Where:
- $y_i^{obs}$ = observed value at location/time $i$
- $y_i^{sim}$ = simulated value at location/time $i$  
- $w_i$ = weight assigned to observation $i$
- $n$ = number of observations

**Weighting** is crucial: observations with higher confidence or importance get larger weights.

#### Parameters and Parameter Groups

Parameters can be defined as:
- **Single values**: e.g., one K value for the entire aquifer
- **Zones**: different K values for geological units
- **Pilot points**: spatially distributed points that define a continuous field through interpolation

#### Regularization

With many parameters, models can become **non-unique** (many parameter combinations fit the data equally well). Regularization adds constraints:

- **Tikhonov regularization**: Penalizes deviation from preferred parameter values
- **Subspace regularization (SVD-Assist)**: Reduces parameter space dimensionally
- **Prior information**: Incorporates expert knowledge as soft constraints

#### Sensitivity Analysis

Before calibration, **sensitivity analysis** identifies:
- Which parameters influence model outputs most
- Which parameters can be estimated from available data
- Correlations between parameters (identifiability issues)

### 10.4 - pyEMU: Python Interface to PEST++

**pyEMU** (Python for Environmental Model Uncertainty) is a Python package that creates PEST++ input files directly from FloPy models. It dramatically simplifies the setup process.

#### Installation

```bash
# Install pyEMU (includes PEST++ executables)
pip install pyemu

# Or with conda
conda install -c conda-forge pyemu
```

#### The PEST++ Workflow with pyEMU

The calibration workflow is **inherently iterative**. You rarely get optimal results on the first attempt - instead, you refine your setup based on results from each iteration:

```
                    ┌──────────────────────────────────────────────┐
                    │           ITERATIVE REFINEMENT LOOP          │
                    │                                              │
                    ▼                                              │
┌─────────────────┐     ┌─────────────────┐     ┌─────────────────┐│
│   FloPy Model   │────▶│  pyEMU Setup    │────▶│  PEST++ Files   ││
│   (*.nam, etc)  │     │  (PstFrom)      │     │  (*.pst, *.tpl) ││
└─────────────────┘     └─────────────────┘     └─────────────────┘│
                                                        │          │
                                                        ▼          │
┌─────────────────┐     ┌─────────────────┐     ┌─────────────────┐│
│   Analyze &     │◀────│  Post-process   │◀────│  Run pestpp-glm ││
│   Decide        │     │  Results        │     │  or pestpp-ies  ││
└─────────────────┘     └─────────────────┘     └─────────────────┘│
        │                                                          │
        │  Questions to ask after each iteration:                  │
        │  • Are residuals acceptable? If not, why?                │
        │  • Are parameters hitting bounds? Adjust bounds?         │
        │  • Are there identifiability issues? Add regularization? │
        │  • Should I add/remove parameters or observations?       │
        │                                                          │
        └──────────────────────────────────────────────────────────┘
```

> **Note**: A typical calibration project involves 5-20 iterations of refinement before achieving satisfactory results. Each iteration teaches you something about your model and data!

### 10.5 - Example: Setting Up PEST++ with pyEMU

The following cells demonstrate how to set up automatic calibration for our Limmat Valley model. This is a **conceptual example** - in practice, you would need PEST++ installed and more observation data for robust calibration.

#### Step 1: Define the PEST++ Setup with PstFrom

The `PstFrom` class is pyEMU's main interface for building PEST++ problems from existing models.

In [ ]:
# Example: Setting up PEST++ with pyEMU (conceptual - requires pyemu installation)
# 
# This code demonstrates the workflow but won't run without pyemu installed.
# To install: pip install pyemu

PYEMU_AVAILABLE = False
try:
    import pyemu
    PYEMU_AVAILABLE = True
    print(f"pyEMU version {pyemu.__version__} is available!")
except ImportError:
    print("pyEMU is not installed. The following cells show the workflow conceptually.")
    print("To install: pip install pyemu")

In [ ]:
# Step 1: Create a PEST++ setup directory and initialize PstFrom
# This copies model files and prepares the calibration workspace

if PYEMU_AVAILABLE:
    # Create a new directory for the PEST++ setup
    pest_ws = os.path.join(workspace, 'pest_calibration')
    
    # Initialize PstFrom - this is the main pyEMU class for building PEST++ problems
    pf = pyemu.utils.PstFrom(
        original_d=workspace,           # Directory with original model files
        new_d=pest_ws,                   # Directory for PEST++ setup
        remove_existing=True,            # Remove existing pest_ws if it exists
        longnames=True,                  # Use long parameter/observation names
        spatial_reference=mf.modelgrid,  # Pass the model grid for spatial operations
    )
    print(f"PstFrom initialized. PEST++ workspace: {pest_ws}")
else:
    print("""
# Conceptual code - what this would do:
# ─────────────────────────────────────
pest_ws = os.path.join(workspace, 'pest_calibration')

pf = pyemu.utils.PstFrom(
    original_d=workspace,           # Directory with original model files
    new_d=pest_ws,                   # Directory for PEST++ setup  
    remove_existing=True,            # Clean start
    longnames=True,                  # Use descriptive names
    spatial_reference=mf.modelgrid,  # For spatial parameter setup
)
""")

#### Step 2: Define Adjustable Parameters

We specify which model inputs PEST++ can adjust during calibration. Common choices:
- **Hydraulic conductivity (K)** - as zones, pilot points, or grid-based
- **Recharge** - spatially variable or uniform multiplier
- **River conductance** - often as a multiplier
- **Storage parameters** - for transient models

In [ ]:
# Step 2: Add parameters for calibration

if PYEMU_AVAILABLE:
    # Option A: Add hydraulic conductivity as a grid-based parameter
    # This creates a multiplier for each active cell (can be many parameters!)
    
    # First, get the HK array filename from the UPW package
    hk_file = f"{model_name}.upw_hk_layer1.txt"  # Adjust based on your model
    
    # Add K as adjustable parameters with bounds
    pf.add_parameters(
        filenames=hk_file,
        par_type="grid",           # Each cell gets a parameter
        par_name_base="hk",        # Parameter name prefix
        pargp="hk_layer1",         # Parameter group name
        lower_bound=0.1,           # Minimum K (m/day)
        upper_bound=500.0,         # Maximum K (m/day)
        transform="log",           # Log-transform (K varies over orders of magnitude)
    )
    
    # Option B: Add recharge as a constant multiplier (simpler)
    rch_file = f"{model_name}.rch_rech.txt"  # Adjust based on your model
    
    pf.add_parameters(
        filenames=rch_file,
        par_type="constant",       # Single multiplier for entire array
        par_name_base="rch_mult",
        pargp="recharge",
        lower_bound=0.5,           # Allow 50% reduction
        upper_bound=2.0,           # Allow 100% increase
        transform="none",
    )
    
    print("Parameters added to PEST++ setup")
    
else:
    print("""
# Conceptual code for adding parameters:
# ──────────────────────────────────────

# Option A: Grid-based hydraulic conductivity (many parameters)
pf.add_parameters(
    filenames="model.upw_hk_layer1.txt",
    par_type="grid",           # Each active cell = 1 parameter
    par_name_base="hk",        
    pargp="hk_layer1",         # Group for reporting
    lower_bound=0.1,           # Physical minimum
    upper_bound=500.0,         # Physical maximum  
    transform="log",           # Log-transform recommended for K
)

# Option B: Recharge multiplier (single parameter)
pf.add_parameters(
    filenames="model.rch_rech.txt",
    par_type="constant",       # One multiplier for whole array
    par_name_base="rch_mult",
    pargp="recharge",
    lower_bound=0.5,
    upper_bound=2.0,
    transform="none",
)

# Option C: Pilot points (spatially varying, fewer parameters than grid)
# Pilot points are interpolated to create smooth parameter fields
pf.add_parameters(
    filenames="model.upw_hk_layer1.txt",
    par_type="pilotpoints",    # Distributed points, interpolated
    par_name_base="hk_pp",
    pargp="hk_pilot",
    pp_space=500,              # Spacing between pilot points (m)
    lower_bound=0.1,
    upper_bound=500.0,
    transform="log",
)
""")

#### Step 3: Define Observations

Observations are the calibration targets - values that PEST++ tries to match. We add our head observations with appropriate weights.

In [ ]:
# Step 3: Add observations for calibration

if PYEMU_AVAILABLE:
    # Add head observations from model output file
    # pyEMU will create instruction files to read simulated values
    
    head_file = f"{model_name}.hds"
    
    # For each observation well, we need to specify:
    # - Location (row, col, layer) in the model
    # - Observed value
    # - Weight (higher = more important in calibration)
    
    # Create observation data from our comparison_df
    if 'comparison_df' in dir() and len(comparison_df) > 0:
        for _, row in comparison_df.iterrows():
            pf.add_observations(
                filename=head_file,
                insfile=f"{row['well_id']}.ins",  # Instruction file
                index_cols=["layer", "row", "col"],
                use_cols=["head"],
                prefix=f"head_{row['well_id']}",
                obsgp="head_obs",
            )
    
    print("Observations added to PEST++ setup")
    
else:
    print("""
# Conceptual code for adding observations:
# ────────────────────────────────────────

# pyEMU can read observations directly from model output files
# You specify where in the file to find simulated values

# Method 1: Using FloPy's built-in observation handling
pf.add_observations(
    filename="model.hds",           # Model output file
    insfile="heads.ins",             # Instruction file (created by pyEMU)
    index_cols=["layer", "row", "col"],
    use_cols=["head"],
    prefix="head",
    obsgp="head_obs",                # Observation group
)

# Method 2: Custom observations from a CSV
# If you have observed values in a separate file:
obs_df = pd.DataFrame({
    'well_id': ['3601', '3625', '83-1'],
    'observed': [400.46, 400.54, 400.92],
    'weight': [1.0, 1.0, 1.0],          # Equal weights
    'row': [32, 18, 10],
    'col': [134, 133, 138],
})

# Weights can be based on:
# - Measurement uncertainty (weight = 1/std_dev)
# - Importance to calibration objectives
# - Data quality
""")

#### Step 4: Build the PEST Control File and Run

The final step generates all PEST++ input files and (optionally) runs the calibration.

In [ ]:
# Step 4: Build PEST++ control file and run calibration

if PYEMU_AVAILABLE:
    # Build the PEST control file (.pst)
    # This generates all template files, instruction files, and the control file
    pst = pf.build_pst()
    
    # Review what was created
    print(f"Number of parameters: {pst.npar}")
    print(f"Number of observations: {pst.nobs}")
    print(f"Parameter groups: {pst.par_groups}")
    print(f"Observation groups: {pst.obs_groups}")
    
    # Set PEST++ control options
    pst.control_data.noptmax = 20       # Maximum optimization iterations
    pst.pestpp_options["ies_par_en"] = "prior.jcb"  # For uncertainty analysis
    
    # Write the control file
    pst.write(os.path.join(pest_ws, "calibration.pst"))
    print(f"\nPEST++ control file written to: {pest_ws}/calibration.pst")
    
else:
    print("""
# Conceptual code for building and running PEST++:
# ─────────────────────────────────────────────────

# Build the PEST control file
pst = pf.build_pst()

# Review the setup
print(f"Parameters: {pst.npar}")
print(f"Observations: {pst.nobs}")

# Configure PEST++ options
pst.control_data.noptmax = 20           # Max iterations
pst.pestpp_options["ies_num_reals"] = 100   # Ensemble size for IES

# Write control file
pst.write(os.path.join(pest_ws, "calibration.pst"))

# ─────────────────────────────────────────────────
# To run PEST++ from command line:
# ─────────────────────────────────────────────────

# For gradient-based calibration:
#   pestpp-glm calibration.pst

# For ensemble-based calibration with uncertainty:
#   pestpp-ies calibration.pst

# For parallel runs (using 4 workers):
#   pestpp-glm calibration.pst /h :4

# ─────────────────────────────────────────────────
# Or run from Python:
# ─────────────────────────────────────────────────

# pyemu.os_utils.run("pestpp-glm calibration.pst", cwd=pest_ws)
""")

### 10.6 - Post-Processing Calibration Results

After PEST++ runs, pyEMU provides tools to analyze results, visualize parameter changes, and assess uncertainty.

In [ ]:
# Post-processing PEST++ results (conceptual example)

print("""
After running PEST++, you can analyze results with pyEMU:

# ─────────────────────────────────────────────────
# Load calibration results
# ─────────────────────────────────────────────────

# Load the completed PEST control file with results
pst = pyemu.Pst(os.path.join(pest_ws, "calibration.pst"))

# Read the residuals file (observed vs simulated)
res_df = pyemu.pst_utils.read_resfile(os.path.join(pest_ws, "calibration.rei"))

# ─────────────────────────────────────────────────
# Analyze objective function progress
# ─────────────────────────────────────────────────

# Read iteration record
iobj = pd.read_csv(os.path.join(pest_ws, "calibration.iobj"))
plt.plot(iobj.iteration, iobj.total_phi)
plt.xlabel("Iteration")
plt.ylabel("Objective Function (Φ)")
plt.title("Calibration Progress")

# ─────────────────────────────────────────────────
# Compare initial vs calibrated parameters
# ─────────────────────────────────────────────────

# Parameter summary
par_df = pst.parameter_data
print("Calibrated parameters:")
print(par_df[['parnme', 'parval1', 'parlbnd', 'parubnd']])

# Check which parameters hit bounds (potential issues!)
at_bounds = par_df[(par_df.parval1 <= par_df.parlbnd * 1.01) | 
                    (par_df.parval1 >= par_df.parubnd * 0.99)]
if len(at_bounds) > 0:
    print("WARNING: Parameters at bounds - consider expanding bounds")

# ─────────────────────────────────────────────────
# Residual analysis
# ─────────────────────────────────────────────────

# Scatter plot of simulated vs observed
plt.scatter(res_df.measured, res_df.modelled)
plt.plot([res_df.measured.min(), res_df.measured.max()],
         [res_df.measured.min(), res_df.measured.max()], 'k--')
plt.xlabel("Observed")
plt.ylabel("Simulated")

# Calculate final metrics
rmse = np.sqrt((res_df.residual**2).mean())
print(f"Final RMSE: {rmse:.2f} m")
""")

### 10.7 - Choosing Between PEST++ Programs

| Use Case | Recommended Program | When to Use |
|----------|--------------------|----|
| **Standard calibration** | `pestpp-glm` | Few to moderate parameters, well-posed problem |
| **Uncertainty analysis** | `pestpp-ies` | Need parameter/prediction uncertainty, many parameters |
| **Sensitivity analysis** | `pestpp-sen` | Identify important parameters before calibration |
| **Highly parameterized** | `pestpp-ies` | 1000s of parameters (e.g., pilot points) |
| **Constrained optimization** | `pestpp-opt` | Management optimization with constraints |

### 10.8 - Best Practices for Automatic Calibration

1. **Start simple**: Begin with few parameters, add complexity iteratively
2. **Check identifiability**: Not all parameters can be estimated from available data
3. **Use regularization**: Prevents overfitting and ensures physical plausibility
4. **Validate independently**: Reserve data for validation (not used in calibration)
5. **Document everything**: Keep records of each calibration iteration
6. **Physical bounds**: Always constrain parameters to realistic ranges
7. **Multiple observation types**: Combine heads, fluxes, concentrations when available

### 10.9 - Common Pitfalls

| Problem | Symptoms | Solution |
|---------|----------|----------|
| **Non-uniqueness** | Many parameter sets fit equally well | Add regularization, more observation types |
| **Insensitive parameters** | Parameters don't change during calibration | Fix at reasonable values or remove |
| **Correlated parameters** | Parameters trade off against each other | Reduce parameterization, use prior information |
| **Overfitting** | Excellent calibration, poor validation | Use regularization, cross-validation |
| **Local minima** | Results depend on starting values | Try multiple starting points, use global methods |

### 10.10 - Resources for Learning PEST++ and pyEMU

#### Official Documentation & Downloads

| Resource | URL | Description |
|----------|-----|-------------|
| **PEST++ GitHub** | https://github.com/usgs/pestpp | Source code, releases, documentation |
| **pyEMU GitHub** | https://github.com/pypest/pyemu | Python package for PEST++ |


## Next Steps

In the next notebook (6_validation.ipynb), we will:
1. Test the calibrated model against reserved validation data
2. Assess the model's predictive capability
3. Discuss when recalibration might be needed

## References

### Groundwater Modeling & Calibration (Free Resources)

- Hill, M.C. (1998). *Methods and Guidelines for Effective Model Calibration*. U.S. Geological Survey Water-Resources Investigations Report 98-4005. **[Free PDF](https://water.usgs.gov/nrp/gwsoftware/modflow2000/WRIR98-4005.pdf)**

- Reilly, T.E. & Harbaugh, A.W. (2004). *Guidelines for Evaluating Ground-Water Flow Models*. U.S. Geological Survey Scientific Investigations Report 2004-5038. **[Free PDF](https://pubs.usgs.gov/sir/2004/5038/PDF/SIR20045038part2.pdf)**


### PEST & PEST++ (Free Resources)

- Doherty, J. (2015). *Calibration and Uncertainty Analysis for Complex Environmental Models*. Watermark Numerical Computing. **[Free PDF](https://pesthomepage.org/pest-book)**

- White, J.T., Hunt, R.J., Fienen, M.N., & Doherty, J.E. (2020). *Approaches to highly parameterized inversion: PEST++ Version 5*. U.S. Geological Survey Techniques and Methods 7-C26. **[Free PDF](https://pubs.usgs.gov/publication/tm7C26)**

- GMDSI (Groundwater Modelling Decision Support Initiative) - Free tutorials and courses on calibration and uncertainty analysis. **[Website](https://gmdsi.org/education/tutorials/)**

### pyEMU (Free Resources)

- White, J.T., Fienen, M.N., & Doherty, J.E. (2016). "A python framework for environmental model uncertainty analysis." *Environmental Modelling & Software*, 85, 217-228. **[GitHub Repository](https://github.com/pypest/pyemu)**

### Additional Reading (Commercial)

- Anderson, M.P., Woessner, W.W., Hunt, R.J. (2015). *Applied Groundwater Modeling* (2nd ed.). Academic Press. *(Comprehensive textbook - library access recommended)* [https://www.sciencedirect.com/book/monograph/9780120581030/applied-groundwater-modeling](https://www.sciencedirect.com/book/monograph/9780120581030/applied-groundwater-modeling)

- White, J.T., et al. (2020). "Towards improved environmental modeling outcomes: Enabling low-cost access to high-dimensional, geostatistical-based decision-support analyses." *Environmental Modelling & Software*. [https://doi.org/10.1016/j.envsoft.2021.105022](https://doi.org/10.1016/j.envsoft.2021.105022)

- White, J.T. (2018). "A model-independent iterative ensemble smoother for efficient history-matching and uncertainty quantification in very high dimensions." *Environmental Modelling & Software*. [https://doi.org/10.1016/j.envsoft.2018.06.009](https://doi.org/10.1016/j.envsoft.2018.06.009)